**Goal**: Explore Deep Agent with Agent Skills

To build an agent with skills that helps user in improving a prompt(eg system prompt)

**Install packages**

In [ ]:
!pip install -qU deepagents
!pip install -U "langchain[google-genai]"

**Import packages**

In [2]:
import os
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

**Set google gemini api key**

In [4]:
os.environ["GEMINI_API_KEY"] = "GEMINI_API_KEY"

**Create a deep agent with FilesystemBackend by giving access to root folder containing prompt improver skill.md file.**

In [ ]:
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import MemorySaver
from deepagents.backends.filesystem import FilesystemBackend

# Checkpointer is REQUIRED for human-in-the-loop
checkpointer = MemorySaver()

agent = create_deep_agent(
    model="google_genai:gemini-2.5-flash",
    backend=FilesystemBackend(root_dir="/content/skills"),
    skills=["/content/skills/"],
    interrupt_on={
        "write_file": True,  # Default: approve, edit, reject
        "read_file": False,  # No interrupts needed
        "edit_file": True    # Default: approve, edit, reject
    },
    checkpointer=checkpointer,  # Required!
)

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """can you improve this prompt
You are a Sustainability Advisor. Your task is to help users understand the climate impact of their lifestyle choices.

You MUST use the available tool `climate_impact_compass` to classify each choice into the Climate Impact Compass:
- X-axis: Easy to Change (-1) <-> Hard to Change (+1)
- Y-axis: Low CO₂ Footprint (-1) <-> High CO₂ Footprint (+1)

⚠️ Important Rules:
1. Only two values are allowed per axis: -1 or +1 (no "moderate", "medium", or decimals).
2. Always call the tool to create classifications and generate the visualization.
""",
            }
        ]
    },
    config={"configurable": {"thread_id": "12345"}},
)

The agent has access to SKILLS folder which follows agent skills folder structure. Testing with simple system prompt and asking agent to improve it.

**Agent flow**

In [6]:
for m in result['messages']:
    m.pretty_print()

================================ Human Message =================================

can you improve this prompt
You are a Sustainability Advisor. Your task is to help users understand the climate impact of their lifestyle choices.

You MUST use the available tool `climate_impact_compass` to classify each choice into the Climate Impact Compass:
- X-axis: Easy to Change (-1) <-> Hard to Change (+1)
- Y-axis: Low CO₂ Footprint (-1) <-> High CO₂ Footprint (+1)

⚠️ Important Rules:
1. Only two values are allowed per axis: -1 or +1 (no "moderate", "medium", or decimals).
2. Always call the tool to create classifications and generate the visualization.

================================== Ai Message ==================================
Tool Calls:
  read_file (f8ce0e37-498d-4f84-95c6-d5f73020c931)
 Call ID: f8ce0e37-498d-4f84-95c6-d5f73020c931
  Args:
    file_path: /content/skills/prompt-improver/SKILL.md
================================= Tool Message =================================
Name: read_

The agent reads the skills file using read tool before acting

**The agent was able to successfully improve the prompt after accessing skills file**

In [7]:
print(result["messages"][-1].content[0]["text"])

Improved Prompt:
You are an expert Sustainability Advisor, specializing in evaluating the environmental impact of lifestyle choices. Your primary goal is to provide users with clear, actionable insights into their carbon footprint using the `climate_impact_compass` tool.

For each lifestyle choice presented by the user, you MUST classify it across two dimensions using the `climate_impact_compass` tool:

**X-axis: Ease of Change**
*   **-1 (Easy to Change):** Actions that can be adopted or altered with minimal effort, cost, or disruption to daily routine (e.g., switching off lights, reducing food waste).
*   **+1 (Hard to Change):** Actions requiring significant effort, financial investment, lifestyle adjustments, or long-term commitment (e.g., buying an electric car, renovating a home for energy efficiency).

**Y-axis: CO₂ Footprint**
*   **-1 (Low CO₂ Footprint):** Choices that inherently produce minimal greenhouse gas emissions (e.g., walking, consuming local produce in season).
*   

**Asking agent which skill was used using persistent memory across threads**

In [ ]:
result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": """ which skill did you last time?
""",
            }
        ]
    },
    config={"configurable": {"thread_id": "12345"}},
)

In [9]:
print(result["messages"][-1].content[0]["text"])

I used the `prompt-improver` skill.


**Take away**:

Skills are reusable agent capabilities that provide specialized workflows and domain knowledge. Can use Agent Skills to provide deep agent with new capabilities and expertise.